## Preprocess data

This notebook expects to take standard input data from the corridor model and analytics pipelines and output tables that are ready for the front end to use. Most notably this includes creating regional clusters and tagging each asset to them

### Import necessary functions

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os

### Little hack to reproducibly set working dir as the project directory (one level up) in notebooks
if "original_dir" not in vars():
    original_dir = os.path.abspath("")

os.chdir(original_dir)
os.chdir("..")  ### Adjust as needed to get to root
print(f"Project root (ensure this is correct): {os.getcwd()}")

import shared.preprocessing.data_preprocessing as funcs
import shared.cleaning_functions.general_cleaning_functions as general_cleaning_functions
import pandas as pd
import re
from shared.io_utils import io_utils
from sqlalchemy import create_engine
from sqlalchemy.engine import URL
import sqlite3

pd.set_option("display.max_columns", 999)
from app.src.config import BASE_PATH, DB_SERVER, DB_PORT, DB_NAME, DB_USERNAME, DB_PASSWORD

Project root (ensure this is correct): /Users/Asha_Pareek/Library/CloudStorage/OneDrive-McKinsey&Company/Desktop/vegx_thailand_front_end


### Load files

In [3]:
if "engine" not in globals():
    general_cleaning_functions.log_detail("Initialize database engine.")
    
    # Create URL object
    connection_url = URL.create(
    "mssql+pyodbc",
    username=DB_USERNAME,
    password=DB_PASSWORD,
    host=DB_SERVER,
    port=DB_PORT,
    database=DB_NAME,
    query={
        "driver": "ODBC Driver 17 for SQL Server",
        "TrustServerCertificate": "yes",
        "Encrypt": "no",
        "timeout": "120",
    }
    )

    # Create engine with modified settings
    engine = create_engine(
        connection_url,
        pool_size=5,
        max_overflow=10,
    )

else:
    general_cleaning_functions.log_detail("Reusing the existing database engine.")

	Initialize database engine.


In [4]:
engine.connect()

In [5]:
aoj, _= io_utils.read_from_db("areas_of_jurisdiction", "corridor_model", engine)
corridors_df, _= io_utils.read_from_db("front_end_input", "risk_modeling", engine)
scenario_outcomes, _= io_utils.read_from_db("optimization_scenarios", "risk_modeling", engine)
scenario_features, _= io_utils.read_from_db("optimization_metrics", "risk_modeling", engine)
shap_features, _= io_utils.read_from_db("model_shap_features", "risk_modeling", engine)
values_df, _= io_utils.read_from_db("model_inputs_prediction", "risk_modeling", engine)

In [10]:
corridors_df.head()

,nearest_upstream_device,feeder_id_traced,line_type,corridor_length,number_lines,geometry,outages_lag_1,outages_lag_2,outages_lag_3,density_mean,nearest_upstream_device_list,health_score,adjusted_area_customers,density_distribution_meta,trimming_cost_meta,trimming_cost_mjm,raw_customers,criticality,risk
0,KPA01VF-31-1262SW000000002,KPA01,overhead,8.983837,2,"MULTILINESTRING ((527269.865 1842217.086, 5272...",0.0,0.0,0.0,0.0,None,0.029555,1.000000,{'open': '0.01 km'},1.374527,0.00000,1.0,1.000000,0.017467
1,BOA07F-217-42SWKA000018864,BOA07,overhead,26.567476,1,"MULTILINESTRING ((534853.343 999708.651, 53486...",0.0,0.0,0.0,0.0,None,0.024786,1.052780,{'open': '0.03 km'},3.905419,59.98936,1.0,1.052780,0.018389
2,CAY01F-205-42SWKA000023142,CAY01,overhead,10.811573,2,"MULTILINESTRING ((514565.573 1055937.917, 5145...",0.0,0.0,0.0,0.0,None,0.029020,1.000000,{'open': '0.01 km'},1.589301,31.20792,1.0,1.000000,0.017467
3,BOA05F-223-42SWKA000025723,PPA04,overhead,12.807387,1,"MULTILINESTRING ((534239.765 1003750.324, 5342...",0.0,0.0,0.0,0.0,None,0.024786,1.146580,{'open': '0.01 km'},1.882686,0.00000,1.0,1.146580,0.020028
4,BOA05F-212-42SWKA000033121,BOA05,overhead,8.255754,1,"MULTILINESTRING ((534425.023 1005900.491, 5344...",0.0,0.0,0.0,0.0,None,0.024786,1.325538,{'open': '0.01 km'},1.213596,0.00000,1.0,1.325538,0.023154


In [6]:
# To get the Main/Lateral line tag
corridors_lines, _= io_utils.read_from_db("input_spans_to_corridors", "corridor_model", engine)

### Load parameters

In [7]:
raw_data_processing_parameters = io_utils.load_param("raw_data_processing_parameters")
local_path = io_utils.load_param("local_path")

### Process Data

In [8]:
full_corridor_name = corridors_df[
    ["nearest_upstream_device", "full_nearest_upstream_device"]
]
corridors_df = corridors_df.drop("nearest_upstream_device", axis=1).rename(
    columns={"full_nearest_upstream_device": "nearest_upstream_device"}
)
scenario_outcomes = scenario_outcomes.rename(
    columns={"nearest_upstream_device": "nearest upstream device"}
)
shap_features = shap_features.rename(
    columns={"full_nearest_upstream_device": "nearest upstream device"}
)
values_df = values_df.drop("nearest_upstream_device", axis=1).rename(
    columns={"full_nearest_upstream_device": "nearest upstream device"}
)
shap_features = funcs.map_and_rename_nearest_upstream_device(
    df=shap_features, corridors_df=full_corridor_name
)
scenario_outcomes = funcs.map_and_rename_nearest_upstream_device(
    df=scenario_outcomes, corridors_df=full_corridor_name
)
scenario_features["scenario"] = scenario_features["scenario"].replace(
    {
        "scenario_1": "Reliability Focus Scenario",
        "scenario_2": "Cost Focus Scenario",
        "scenario_chosen": "Chosen Scenario",
    }
)
aoj["NAME"] = aoj["NAME"].map(lambda x: re.sub(r"\.", "", x))
corridors_df = funcs.prepare_corridors(corridors_df)
veg_df = funcs.create_vegetation_features(
    corridors_df,
    raw_data_processing_parameters["columns_to_categorize"],
    raw_data_processing_parameters["column_renames"],
)
scenario_outcomes = funcs.prepare_scenario_outcomes(scenario_outcomes)
veg_df = funcs.merge_scenario_outcomes(veg_df, scenario_outcomes)

In [9]:
veg_df.head()

,nearest upstream device,feeder id,line type,corridor length,number lines,geometry,count of outages last year,count of outages last two years,count of outages last three years,average vegetation density (%) [META + Sentinel-2],nearest upstream device list,probability of outage (%),customers affected (adjusted),density_distribution_meta,cost to trim (BHT) [META + Sentinel-2],cost to trim (BHT) [MJM],customers affected (original),criticality,risk (customer interruptions),corridor length (km),probability of outage (%) [bins],risk (customer interruptions) [bins],customers affected (adjusted) [bins],scenario 1 frequency,scenario 2 frequency,scenario chosen frequency
0,KPA01VF-31-1262SW000000002,KPA01,overhead,8.98,2,"MULTILINESTRING ((527269.865 1842217.086, 5272...",0.0,0.0,0.0,0.0,None,0.03,1.00,{'open': '0.01 km'},1.37,0.00,1.0,1.00,0.02,0.00898,low,low,low,T4,T1,T4
1,BOA07F-217-42SWKA000018864,BOA07,overhead,26.57,1,"MULTILINESTRING ((534853.343 999708.651, 53486...",0.0,0.0,0.0,0.0,None,0.02,1.05,{'open': '0.03 km'},3.91,59.99,1.0,1.05,0.02,0.02657,low,low,low,T4,T1,T4
2,CAY01F-205-42SWKA000023142,CAY01,overhead,10.81,2,"MULTILINESTRING ((514565.573 1055937.917, 5145...",0.0,0.0,0.0,0.0,None,0.03,1.00,{'open': '0.01 km'},1.59,31.21,1.0,1.00,0.02,0.01081,low,low,low,T4,T1,T4
3,BOA05F-223-42SWKA000025723,PPA04,overhead,12.81,1,"MULTILINESTRING ((534239.765 1003750.324, 5342...",0.0,0.0,0.0,0.0,None,0.02,1.15,{'open': '0.01 km'},1.88,0.00,1.0,1.15,0.02,0.01281,low,low,low,T4,T1,T4
4,BOA05F-212-42SWKA000033121,BOA05,overhead,8.26,1,"MULTILINESTRING ((534425.023 1005900.491, 5344...",0.0,0.0,0.0,0.0,None,0.02,1.33,{'open': '0.01 km'},1.21,0.00,1.0,1.33,0.02,0.00826,low,low,low,T4,T1,T4


In [9]:
veg_df = funcs.map_frequencies(
    veg_df,
    raw_data_processing_parameters["frequency_mapping"],
    raw_data_processing_parameters["frequency_columns"],
)

In [10]:
veg_out = funcs.prepare_output_dataframe(
    veg_df,
    raw_data_processing_parameters["columns_to_drop"],
    raw_data_processing_parameters["output_columns"],
    raw_data_processing_parameters["rename_columns"],
)

In [11]:
veg_df = funcs.calculate_weighted_metrics(veg_df)

In [12]:
veg_df = funcs.general_clean(veg_df)

In [13]:
veg_units = funcs.create_veg_units(
    veg_df,
    aoj,
    corridors_df.crs,
    raw_data_processing_parameters["agg_columns"],
    raw_data_processing_parameters["veg_units_columns"],
)

In [14]:
veg_out_sql, veg_units_sql = funcs.prepare_for_sql(veg_out, veg_units)

/Users/Asha_Pareek/Library/CloudStorage/OneDrive-McKinsey&Company/Desktop/vegx_thailand_front_end/shared/preprocessing/data_preprocessing.py:97: UserWarning: Geometry column does not contain geometry.
  veg_units_sql["polygon"] = veg_units_sql["polygon"].map(lambda x: x.wkt)


### Save

In [15]:
veg_out = veg_out.rename(columns={'nearest upstream device list': 'downstream device list'})
veg_out.head(2)

,feeder id,nearest upstream device,downstream device list,geometry,corridor length (km),density distribution meta,vegetation density (%) [META + Sentinel-2],probability of outage (%),probability of outage (%) [bins],cost to trim (BHT) [META + Sentinel-2],cost to trim (BHT) [MJM],risk (customer interruptions),risk (customer interruptions) [bins],customers affected (original),customers affected (adjusted),customers affected (adjusted) [bins],cost focus scenario frequency,reliability focus scenario frequency,chosen scenario frequency
0,KPA01,KPA01VF-31-1262SW000000002,None,"MULTILINESTRING ((527269.865 1842217.086, 5272...",0.00898,{'open': '0.01 km'},0.0,0.03,low,1.37,0.00,0.02,low,1.0,1.00,low,three times a year,one time a year,three times a year
1,BOA07,BOA07F-217-42SWKA000018864,None,"MULTILINESTRING ((534853.343 999708.651, 53486...",0.02657,{'open': '0.03 km'},0.0,0.02,low,3.91,59.99,0.02,low,1.0,1.05,low,three times a year,one time a year,three times a year


In [16]:
io_utils.write_to_db(veg_out, "veg_features", "front_end_visualization", engine, "replace")
io_utils.write_to_db(veg_units, "veg_clusters", "front_end_visualization", engine, "replace")

/Users/Asha_Pareek/Library/CloudStorage/OneDrive-McKinsey&Company/Desktop/vegx_thailand_front_end/shared/io_utils/io_utils.py:72: UserWarning: Geometry column does not contain geometry.
  df[column] = df[column].apply(lambda x: x.wkt if x is not None else None)
/Users/Asha_Pareek/Library/CloudStorage/OneDrive-McKinsey&Company/Desktop/vegx_thailand_front_end/shared/io_utils/io_utils.py:72: UserWarning: Geometry column does not contain geometry.
  df[column] = df[column].apply(lambda x: x.wkt if x is not None else None)


In [17]:
os.makedirs(os.path.dirname(local_path), exist_ok=True)
conn = sqlite3.connect(local_path + '/digital_twin_database.sqlite')
veg_out_sql.to_sql("veg_features", conn, if_exists="replace", index=False)
veg_units_sql.to_sql("veg_units", conn, if_exists="replace", index=False)
scenario_features.to_sql("veg_scenarios", conn, if_exists="replace", index=False)
shap_features.to_sql("veg_shap_features", conn, if_exists="replace", index=False)
values_df[list(shap_features)].to_sql("veg_raw", conn, if_exists="replace", index=False)
aoj[['NAME', 'Province', 'จังหวัด', 'District', 'เขต']].to_sql("areas_of_jurisdiction", conn, if_exists="replace", index=False)

314

In [ ]:
# import sqlite3
# import os

# db_path = os.path.join(local_path, "digital_twin_database.sqlite")

# # Connect to the SQLite database
# conn = sqlite3.connect(db_path)
# cursor = conn.cursor()

# # Check if the table exists
# cursor.execute("SELECT name FROM sqlite_master WHERE type='table' AND name='areas_of_jurisdiction';")
# table_exists = cursor.fetchone()

# if table_exists:
#     print("Table 'areas_of_jurisdiction' exists.")
# else:
#     print("Table 'areas_of_jurisdiction' does NOT exist.")

# # Close connection
# conn.close()

Table 'areas_of_jurisdiction' exists.
